# Notebook 05 — Minh họa nguyên lý LLaVA-OneVision (Chương 3)

Mở mô hình ra để kiểm chứng bằng số liệu thật các bước đã trình bày ở Chương 3:

| Mục | Kiểm chứng | Liên hệ báo cáo |
|---|---|---|
| 3 | Ảnh được chia thành bao nhiêu ô (AnyRes) | Mục 3.3, Bước 2 ở mục 3.8 |
| 4 | Kích thước đặc trưng sau SigLIP và sau bộ chiếu MLP | Bảng 3.1, mục 3.4–3.5 |
| 5 | Số token ảnh so với số token chỉ dẫn | Mục 3.3, công thức (3.2) |
| 6 | Độ phân giải → số token → bộ nhớ, thời gian | Nhận định ở mục 3.9 |
| 7 | Mô hình chọn từng token như thế nào | Công thức (3.1), mục 3.6 |

**Cần GPU:** Runtime → Change runtime type → T4 GPU.

## 1. Cài đặt và nạp mô hình (4-bit, giống notebook 01)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import io, time, urllib.request
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
PROMPT = "Describe this image in one short sentence."
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
cfg = model.config
print("Bộ mã hóa ảnh:", cfg.vision_config.model_type, "| ảnh vào", cfg.vision_config.image_size,
      "| patch", cfg.vision_config.patch_size, "| số tầng", cfg.vision_config.num_hidden_layers,
      "| chiều", cfg.vision_config.hidden_size)
print("LLM:", cfg.text_config.model_type, "| số tầng", cfg.text_config.num_hidden_layers,
      "| chiều", cfg.text_config.hidden_size, "| từ vựng", cfg.text_config.vocab_size)
print("Chiến lược AnyRes:", getattr(cfg, "vision_aspect_ratio", None))

## 2. Ảnh minh họa

In [ ]:
def load_url(url):
    with urllib.request.urlopen(url, timeout=30) as response:
        return Image.open(io.BytesIO(response.read())).convert("RGB")


cats = load_url("http://images.cocodataset.org/val2017/000000039769.jpg")
print("Kích thước ảnh gốc:", cats.size)
plt.imshow(cats); plt.axis("off"); plt.show()


def build_inputs(image):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                         return_dict=True, return_tensors="pt").to(model.device, torch.float16)

## 3. AnyRes: ảnh được chia thành bao nhiêu ô

In [ ]:
inputs = build_inputs(cats)
pixel_values = inputs["pixel_values"]
print("pixel_values:", tuple(pixel_values.shape), "→ (ảnh, số ô gồm cả ảnh gốc, kênh, cao, rộng)")
print("Kích thước ảnh mà bộ xử lý ghi nhận (image_sizes):", inputs["image_sizes"].tolist())

crops = pixel_values[0].float().cpu()
fig, axes = plt.subplots(1, len(crops), figsize=(3 * len(crops), 3))
for k, ax in enumerate(axes):
    img = crops[k].permute(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min())
    ax.imshow(img); ax.axis("off")
    ax.set_title("ảnh gốc thu nhỏ" if k == 0 else f"ô {k}", fontsize=9)
plt.show()

## 4. Kích thước đặc trưng sau SigLIP và sau bộ chiếu MLP

In [ ]:
# Gắn "móc" (hook) vào hai mô-đun để ghi lại kích thước đầu ra khi mô hình chạy.
shapes = {}


def find_module(name):
    for module_name, module in model.named_modules():
        if module_name.endswith(name):
            return module_name, module
    raise KeyError(name)


def record(label):
    def hook(module, args, output):
        tensor = output[0] if isinstance(output, (tuple, list)) else getattr(output, "last_hidden_state", output)
        shapes[label] = tuple(tensor.shape)
    return hook


handles = []
for label, name in [("SigLIP (vision_tower)", "vision_tower"), ("Bộ chiếu MLP", "multi_modal_projector")]:
    module_name, module = find_module(name)
    handles.append(module.register_forward_hook(record(label)))
    print("Gắn móc vào:", module_name)

with torch.no_grad():
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
for h in handles:
    h.remove()
for label, shape in shapes.items():
    print(f"{label:22s} {shape}")
print("Kỳ vọng theo Bảng 3.1: SigLIP (số ô, 729, 1152); bộ chiếu (..., 3584)")

## 5. Số token ảnh so với số token chỉ dẫn

In [ ]:
image_token_id = cfg.image_token_index
ids = inputs["input_ids"][0]
n_image = int((ids == image_token_id).sum())
n_total = ids.numel()
print(f"Tổng số token đầu vào: {n_total}")
print(f"  token ảnh          : {n_image}")
print(f"  token văn bản      : {n_total - n_image} (khuôn ChatML + chỉ dẫn)")
print(f"Giới hạn lý thuyết: 729 × (1 + 9) = 7290 token ảnh")

## 6. Độ phân giải → số token → bộ nhớ và thời gian

In [ ]:
rows = []
for side in [384, 768, 1152, 1536, 2304]:
    image = cats.resize((side, int(side * cats.height / cats.width)))
    inp = build_inputs(image)
    n_img = int((inp["input_ids"][0] == image_token_id).sum())
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    start = time.time()
    try:
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=30, do_sample=False)
    except torch.cuda.OutOfMemoryError:
        # Hết bộ nhớ cũng là một kết quả: ghi lại rồi thử kích thước tiếp theo
        print(f"{str(image.size):12s} token ảnh={n_img:5d}  HẾT BỘ NHỚ GPU")
        del inp; torch.cuda.empty_cache()
        continue
    seconds = time.time() - start
    peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    text = processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    rows.append((image.size, inp["pixel_values"].shape[1], n_img, round(peak, 2), round(seconds, 1), text))
    print(f"{str(image.size):12s} ô={inp['pixel_values'].shape[1]:2d}  token ảnh={n_img:5d}  "
          f"bộ nhớ cao nhất={peak:5.2f} GB  thời gian={seconds:4.1f}s  | {text}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
tokens = [r[2] for r in rows]
ax1.plot(tokens, [r[3] for r in rows], "o-", color="#4C78A8", label="Bộ nhớ cao nhất (GB)")
ax1.set_xlabel("Số token ảnh"); ax1.set_ylabel("Bộ nhớ GPU cao nhất (GB)")
ax2 = ax1.twinx()
ax2.plot(tokens, [r[4] for r in rows], "s--", color="#E45756", label="Thời gian (giây)")
ax2.set_ylabel("Thời gian sinh 30 token (giây)")
ax1.set_title("Ảnh hưởng của số token ảnh (AnyRes) đến bộ nhớ và thời gian")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout(); plt.savefig("/content/hinh_token_bo_nho.png", dpi=150); plt.show()

## 7. Sinh từng token: mô hình chọn từ như thế nào (công thức 3.1)

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                         output_scores=True, return_dict_in_generate=True)
tokenizer = processor.tokenizer
for step, scores in enumerate(out.scores):
    probs = torch.softmax(scores[0].float(), dim=-1)
    top = torch.topk(probs, 5)
    chosen = tokenizer.decode(out.sequences[0, inputs["input_ids"].shape[1] + step])
    candidates = ", ".join(f"{tokenizer.decode(i)!r}:{p:.2f}" for p, i in zip(top.values.tolist(), top.indices.tolist()))
    print(f"Bước {step + 1:2d} chọn {chosen!r:12s} | top-5: {candidates}")

## 8. Tải hình minh họa về máy

In [ ]:
from google.colab import files
files.download("/content/hinh_token_bo_nho.png")